# PC-FNO3D NATIONAL v4 — moyenne des graines de C' (validation seulement) (Colab, A100)

**À lancer uniquement avec l'accord écrit d'Ines.** Règle `configs/national/regle_ensemble_cp.json`
(sha256 `271ddc612a6f…`, statut : PROPOSITION soumise à l'accord écrit d'Ines), vérifiée par le carnet avant
tout calcul ; code du dépôt au commit figé `bb991bf3a652`. **Aucun entraînement, aucune lecture du test.**

La règle v4 (`regle_v4_national.json`, `60671e843ee6…`) choisit UN entraînement et ne prévoit pas
d'ensemble : cette règle s'y ajoute sans la modifier.

1. **modèle** : moyenne des graines 42, 43, 44 de C' (`runs/national_1km_v4_Cp_s<graine>`) — u, v moyens, w par
   continuité ; les run_dir des graines ne sont que lus ;
2. **validation** (64 instants de la sélection d'entraînement, vérité d'entraînement v3) : tranches
   0–1 km au-dessus du sol pour la moyenne, chaque graine, la meilleure graine (val_score, fixée avant le
   calcul) et l'ERA5 seule, entrée complète puis 25 % des points d'entrée (DSO5) ;
3. **DSO4** : temps de la passe servie de la moyenne (3 passes sur l'A100), seuil 1 s ;
4. **DSO6** (estimation de validation, pas un verdict) : dispersion entre graines, calibrée sur la
   validation (2 plis de jours), couverture 80 % hors pli et Spearman ;
5. **bilan** : `MyDrive/pcfno3d_national/runs/national_1km_v4_Cp_ens/` (dossier neuf ; une reprise relit le bilan).

Le test de la moyenne ne serait lu qu'avec un **accord écrit séparé d'Ines**, après lecture de ce bilan.
Ressources estimées (non mesurées) : ~12–14 Go de GPU sur 40, ~20–30 min. La session s'arrête à la fin.

## 1. Drive, code au commit figé, règles vérifiées, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "bb991bf3a652b803e9e55d2e536ae65f6f325bfe"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

# regle de la moyenne des graines (PROPOSITION tant qu'Ines ne l'a pas validee) : empreinte inscrite ici
from pcfno3d.regles import verifier
REGLE = CODE / "configs" / "national" / "regle_ensemble_cp.json"
regle, sha_ens = verifier(REGLE)
assert sha_ens == "271ddc612a6f0a39b8ff6eaebfc2f0ea33d4c313f2ff737b0b34a9c261e7dbf7", f"regle de la moyenne modifiee depuis la generation du carnet : {sha_ens}"
print("regle de la moyenne verifiee :", sha_ens[:12], "|", regle["statut"][:120])

## 2. Cache de la vérité v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "pas de GPU : choisir A100 dans Execution > Modifier le type d'execution"
nom_gpu = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{nom_gpu}, {total:.0f} Go")
if "A100" not in nom_gpu:
    print("ATTENTION : le DSO4 de la regle se mesure sur A100 ; sur un autre GPU, le temps n'est qu'indicatif")
# estimation (regle_ensemble_cp.json, non mesuree) : 3 x 1,81 Go de poids + une passe ~5-7 Go ; le script
# ecrit la memoire mesuree (dso4_moyenne.json)

## 4. Moyenne des graines sur la validation, DSO4, dispersion — puis fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
import traceback
from google.colab import runtime
P = regle["parametres"]
RUNS = NAT / "runs"
SORTIE = RUNS / P["sortie"]
try:
    if (SORTIE / "bilan_validation.json").exists():
        print("deja calcule (rien n'est recalcule) :", SORTIE)
    else:
        CKPTS = " ".join(shlex.quote(str(choisir_poids(RUNS / f"{P['prefixe']}_s{g}"))) for g in P["graines"])
        try:
            # validation seulement : la moyenne, chaque graine, l'ERA5 seule ; DSO4 ; dispersion (DSO6)
            lancer("ensemble_graines_cp",
                   f"python -u scripts/ensemble_graines.py -c national_1km_colab --runs {shlex.quote(str(RUNS))} "
                   f"--regle {shlex.quote(str(REGLE))} --checkpoints {CKPTS}")
        except RuntimeError as exc:                 # code 3 = criteres non remplis : bilan ecrit
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    print((SORTIE / "bilan_validation.md").read_text(encoding="utf-8")[:8000])
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()